<a href="https://colab.research.google.com/github/kanna1951693/-Traffic-Demand-Forecasting-Flipkart-Gridlock-2.0-CatBoost-/blob/main/model_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings("ignore")

# ==========================
# LOAD DATA
# ==========================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

# ==========================
# MISSING VALUES
# ==========================
for col in ["RoadType", "Weather"]:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

temp_median = train["Temperature"].median()
train["Temperature"] = train["Temperature"].fillna(temp_median)
test["Temperature"] = test["Temperature"].fillna(temp_median)

# ==========================
# TIME FEATURES
# ==========================
for df in [train, test]:

    parts = df["timestamp"].str.split(":", expand=True)

    df["hour"] = parts[0].astype(int)
    df["minute"] = parts[1].astype(int)

    df["time_bin"] = df["hour"] * 60 + df["minute"]

    df["time_sin"] = np.sin(2 * np.pi * df["time_bin"] / 1440)
    df["time_cos"] = np.cos(2 * np.pi * df["time_bin"] / 1440)

    df["is_peak"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_night"] = df["hour"].isin(range(0, 6)).astype(int)

# ==========================
# SAFE AGGREGATION FEATURES
# ==========================

global_mean = train["demand"].mean()

# geohash mean
day49_geo_mean = (
    train[train["day"] == 49]
    .groupby("geohash")["demand"]
    .mean()
)
geo_mean = train.groupby("geohash")["demand"].mean()

train["geo_mean"] = train["geohash"].map(geo_mean)

train["day49_geo_mean"] = (
    train["geohash"]
    .map(day49_geo_mean)
    .fillna(global_mean)
)

test["geo_mean"] = (
    test["geohash"]
    .map(geo_mean)
    .fillna(global_mean)
)

test["day49_geo_mean"] = (
    test["geohash"]
    .map(day49_geo_mean)
    .fillna(global_mean)
)

# hour mean
hour_mean = train.groupby("hour")["demand"].mean()

train["hour_mean"] = train["hour"].map(hour_mean)
test["hour_mean"] = test["hour"].map(hour_mean).fillna(global_mean)

# time mean
time_mean = train.groupby("time_bin")["demand"].mean()

train["time_mean"] = train["time_bin"].map(time_mean)
test["time_mean"] = test["time_bin"].map(time_mean).fillna(global_mean)

# geohash frequency
freq = train["geohash"].value_counts()

train["geohash_freq"] = train["geohash"].map(freq)
test["geohash_freq"] = test["geohash"].map(freq).fillna(0)

# interaction features
for df in [train, test]:

    df["road_lane"] = (
        df["RoadType"].astype(str)
        + "_"
        + df["NumberofLanes"].astype(str)
    )

    df["geo_road"] = (
        df["geohash"].astype(str)
        + "_"
        + df["RoadType"].astype(str)
    )


# ==========================
# DROP TIMESTAMP
# ==========================
train.drop(columns=["timestamp"], inplace=True)
test.drop(columns=["timestamp"], inplace=True)

# ==========================
# FEATURES
# ==========================
target = "demand"
id_col = "Index"

feature_cols = [
    c for c in train.columns
    if c not in [target, id_col]
]

cat_features = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "road_lane",
    "geo_road",
]

cat_features = [
    c for c in cat_features
    if c in feature_cols
]

for col in cat_features:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)

X = train[feature_cols]
y = train[target]

X_test = test[feature_cols]
# ==========================
# DAY WEIGHTS
# ==========================

sample_weight = np.where(
    train["day"] == 49,
    3,
    1
)

X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(
    X,
    y,
    sample_weight,
    test_size=0.2,
    random_state=42
)
model = CatBoostRegressor(
    depth=7,
    iterations=2000,
    learning_rate=0.03,
    loss_function="RMSE",
    eval_metric="R2",
    random_seed=42,
    early_stopping_rounds=150,
    verbose=200
)

model.fit(
    X_tr,
    y_tr,
    sample_weight=w_tr,
    cat_features=cat_features,
    eval_set=(X_val, y_val)
)

val_preds = model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("\nValidation R2:", r2)
print("Best iteration:", model.best_iteration_)

# ==========================
# FINAL TRAINING
# ==========================
model_final = CatBoostRegressor(
    iterations=model.best_iteration_ + 100,
    depth=7,
    learning_rate=0.03,
    loss_function="RMSE",
    random_seed=42,
    verbose=200
)

model_final.fit(
    X,
    y,
    sample_weight=sample_weight,
    cat_features=cat_features
)

# ==========================
# PREDICTIONS
# ==========================
preds = model_final.predict(X_test)

preds = np.clip(preds, 0, 1)

submission = pd.DataFrame({
    "Index": test[id_col],
    "demand": preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nSubmission saved.")
print(submission.head())
print(submission.describe())

Train shape: (77299, 11)
Test shape : (41778, 10)
0:	learn: 0.0492349	test: 0.0495575	best: 0.0495575 (0)	total: 148ms	remaining: 4m 55s
200:	learn: 0.9366031	test: 0.9303672	best: 0.9303672 (200)	total: 19s	remaining: 2m 50s
400:	learn: 0.9481393	test: 0.9404447	best: 0.9404447 (400)	total: 39s	remaining: 2m 35s
600:	learn: 0.9532991	test: 0.9448156	best: 0.9448156 (600)	total: 57.2s	remaining: 2m 13s
800:	learn: 0.9569405	test: 0.9475809	best: 0.9475809 (800)	total: 1m 17s	remaining: 1m 55s
1000:	learn: 0.9592983	test: 0.9492488	best: 0.9492488 (1000)	total: 1m 35s	remaining: 1m 35s
1200:	learn: 0.9611897	test: 0.9506815	best: 0.9506815 (1200)	total: 1m 55s	remaining: 1m 16s
1400:	learn: 0.9628785	test: 0.9517665	best: 0.9517665 (1400)	total: 2m 13s	remaining: 57.2s
1600:	learn: 0.9642533	test: 0.9526562	best: 0.9526562 (1600)	total: 2m 33s	remaining: 38.2s
1800:	learn: 0.9653255	test: 0.9532471	best: 0.9532471 (1800)	total: 2m 51s	remaining: 19s
1999:	learn: 0.9662179	test: 0.953667

In [3]:
fi = model.get_feature_importance(prettified=True)
print(fi.head(20))

        Feature Id  Importances
0         geo_mean    34.941654
1        road_lane    19.875117
2   day49_geo_mean    10.881118
3         RoadType     6.639100
4        time_mean     4.956754
5         time_cos     3.391393
6        hour_mean     3.259053
7         time_sin     2.847667
8              day     2.109661
9         time_bin     2.043273
10        geo_road     1.773050
11            hour     1.367999
12    geohash_freq     1.349291
13   LargeVehicles     1.115767
14         geohash     1.004313
15   NumberofLanes     0.838962
16         Weather     0.646028
17     Temperature     0.419661
18        is_night     0.324675
19         is_peak     0.092992


In [4]:
submission = pd.DataFrame({
    "Index": test[id_col],
    "demand": preds
})

In [10]:
submission.to_csv(
    "submission_depth.csv",
    index=False
)

In [11]:
import os
print(os.listdir())

['.config', 'submission_depth.csv', 'submission_depth8.csv', 'catboost_info', 'submission.csv', 'train.csv', 'test.csv', 'sample_submission.csv', 'sample_data']


In [12]:
from google.colab import files
files.download("submission_depth.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>